# MuseTalk — realistic talking avatar test (Colab T4 GPU)

Goal: see whether an open-source lip-sync model makes our avatar look like a real person talking (anam.ai-style),
and how fast it is on a GPU.

* Input: the avatar face + a Galaxy Toyota answer in the Gemini voice — both built into cell 4.
* Output: an MP4 of the face speaking that answer + the time it took.

VS Code: **Select Kernel → Colab → New Colab Server → GPU (T4)**. colab.google: **Runtime → Change runtime type → T4 GPU**. Then run the cells in order (first run ≈ 15–20 min: installs + weights).

MuseTalk (github.com/TMElyralab/MuseTalk) needs Python 3.10 + torch 2.0.1 + mmcv 2.0.1, which do not install on
Colab's Python 3.12 — so cell 1 makes a separate Python 3.10 environment with micromamba.
Licence: check the MuseTalk README and each downloaded model's licence before any commercial use.

In [13]:
#@title 1) Python 3.10 environment (micromamba) + GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /usr/local bin/micromamba
!micromamba create -y -q -n mt -c conda-forge python=3.10 > /dev/null
!micromamba run -n mt python --version

name, memory.total [MiB]
Tesla T4, 15360 MiB
Python 3.10.22


In [12]:
#@title 2) MuseTalk code + dependencies (~10 min; safe to run again)
%cd /content
![ -d MuseTalk ] || git clone -q https://github.com/TMElyralab/MuseTalk.git
%cd /content/MuseTalk
!micromamba run -n mt pip install -q torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!micromamba run -n mt pip install -q -r requirements.txt
# mmlab one by one: mmpose needs chumpy, which only builds without pip's build isolation
!micromamba run -n mt pip install -q -U openmim "setuptools<70" wheel
!micromamba run -n mt mim install -q mmengine
!micromamba run -n mt mim install -q "mmcv==2.0.1"
!micromamba run -n mt mim install -q "mmdet==3.1.0"
!micromamba run -n mt pip install -q --no-build-isolation chumpy
!micromamba run -n mt mim install -q "mmpose==1.1.0"
# transformers (Whisper) needs huggingface-hub < 1.0; gdown for the face-parsing weights
!micromamba run -n mt pip install -q "huggingface_hub>=0.23,<1.0" "gdown>=5"
!which ffmpeg || apt-get -qq install -y ffmpeg
!micromamba run -n mt python -c "import torch, mmcv, mmdet, mmpose, transformers, huggingface_hub as h; print('OK torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| mmcv', mmcv.__version__, '| mmpose', mmpose.__version__, '| hub', h.__version__)"

/content
/content/MuseTalk
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
openxlab 0.1.3 requires rich~=13.4.2, but you have rich 15.0.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.27.3 requires rich>=13.8.0, but you have rich 13.4.2 which is incompatible.
  Preparing metadata (pyproject.toml) ... done
/usr/bin/ffmpeg
OK torch 2.0.1+cu118 cuda True | mmcv 2.0.1 | mmpose 1.1.0 | hub 0.30.2


In [14]:
%%writefile /content/get_weights.py
# 3) Model weights (~3 GB; files already there are skipped). MuseTalk's own download_weights.sh uses the retired
# huggingface-cli and "gdown --id", so the same files are fetched here with huggingface_hub and gdown.
import os, urllib.request
from huggingface_hub import hf_hub_download
import gdown

M = "/content/MuseTalk/models"
hf = [
    ("TMElyralab/MuseTalk", "musetalkV15/unet.pth", M),
    ("TMElyralab/MuseTalk", "musetalkV15/musetalk.json", M),
    ("TMElyralab/MuseTalk", "musetalk/pytorch_model.bin", M),
    ("TMElyralab/MuseTalk", "musetalk/musetalk.json", M),
    ("stabilityai/sd-vae-ft-mse", "config.json", f"{M}/sd-vae"),
    ("stabilityai/sd-vae-ft-mse", "diffusion_pytorch_model.bin", f"{M}/sd-vae"),
    ("openai/whisper-tiny", "config.json", f"{M}/whisper"),
    ("openai/whisper-tiny", "pytorch_model.bin", f"{M}/whisper"),
    ("openai/whisper-tiny", "preprocessor_config.json", f"{M}/whisper"),
    ("yzd-v/DWPose", "dw-ll_ucoco_384.pth", f"{M}/dwpose"),
    ("ByteDance/LatentSync", "latentsync_syncnet.pt", f"{M}/syncnet"),
]
for repo, name, d in hf:
    target = os.path.join(d, name)
    if not os.path.exists(target):
        hf_hub_download(repo, name, local_dir=d)
    print("ok", target, os.path.getsize(target) // 1_000_000, "MB")
fp = f"{M}/face-parse-bisent"
os.makedirs(fp, exist_ok=True)
if not os.path.exists(f"{fp}/79999_iter.pth"):
    gdown.download(id="154JgKpzCPW82qINcVieuPH3fZ2e0P812", output=f"{fp}/79999_iter.pth", quiet=True)
if not os.path.exists(f"{fp}/resnet18-5c106cde.pth"):
    urllib.request.urlretrieve("https://download.pytorch.org/models/resnet18-5c106cde.pth", f"{fp}/resnet18-5c106cde.pth")
for f in ("79999_iter.pth", "resnet18-5c106cde.pth"):
    print("ok", f"{fp}/{f}", os.path.getsize(f"{fp}/{f}") // 1_000_000, "MB")
print("all weights present")

Overwriting /content/get_weights.py


In [ ]:
#@title 3b) Download the weights
%cd /content/MuseTalk
!micromamba run -n mt python /content/get_weights.py

In [15]:
#@title 4) Our face and a Galaxy Toyota answer (built into this notebook - works from VS Code and colab.google)
import os, base64
os.makedirs("/content/MuseTalk/data/ours", exist_ok=True)
FACE_JPG = "..."
REPLY_WAV = "..."
open("/content/MuseTalk/data/ours/face.jpg", "wb").write(base64.b64decode(FACE_JPG))
open("/content/MuseTalk/data/ours/reply.wav", "wb").write(base64.b64decode(REPLY_WAV))
print(os.listdir("/content/MuseTalk/data/ours"))

['face.mp4', 'face.jpg', 'reply.wav']


In [18]:
#@title 5) The face as a 25 fps video as long as the answer, then lip-sync it
%cd /content/MuseTalk
import subprocess, time, glob, os
dur = float(subprocess.check_output(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0",
                                     "data/ours/reply.wav"]).decode().strip())
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-loop", "1", "-i", "data/ours/face.jpg", "-t", str(dur), "-r", "25",
                "-vf", "scale=600:800,format=yuv420p", "data/ours/face.mp4"], check=True)
open("configs/inference/ours.yaml", "w").write("task_0:\n  video_path: data/ours/face.mp4\n  audio_path: data/ours/reply.wav\n")
t = time.time()
r = subprocess.run(["micromamba", "run", "-n", "mt", "python", "-m", "scripts.inference",
                    "--inference_config", "configs/inference/ours.yaml", "--result_dir", "results/ours",
                    "--unet_model_path", "models/musetalkV15/unet.pth", "--unet_config", "models/musetalkV15/musetalk.json",
                    "--version", "v15"], capture_output=True, text=True)
took = time.time() - t
print(r.stdout[-3000:])
videos = glob.glob("results/ours/**/*.mp4", recursive=True)
if r.returncode != 0 or not videos:
    print(r.stderr[-4000:])
    raise RuntimeError("MuseTalk did not make a video - the error is above")
print(f"\n{took:.0f} s for {dur:.1f} s of speech  ->  {took / dur:.1f}x real time (live needs < 1)")

/content/MuseTalk

Traceback (most recent call last):
  File "/root/.local/share/mamba/envs/mt/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/root/.local/share/mamba/envs/mt/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/content/MuseTalk/scripts/inference.py", line 2, in <module>
    import cv2
ModuleNotFoundError: No module named 'cv2'



RuntimeError: MuseTalk did not make a video - the error is above

In [17]:
#@title 6) Watch and download
import os
import glob, base64
from IPython.display import HTML
videos = sorted(glob.glob("/content/MuseTalk/results/ours/**/*.mp4", recursive=True), key=os.path.getmtime)
assert videos, "no video yet - run cell 5 first and check its error"
out = videos[-1]
print(out)
HTML(f'<video width=360 controls src="data:video/mp4;base64,{base64.b64encode(open(out, "rb").read()).decode()}"></video>')

AssertionError: no video yet - run cell 5 first and check its error

## What to look at
* **Lips/teeth/cheeks** — do they look like a real person saying the words (Hindi/Hinglish)?
* **Speed** (cell 5): offline ×real-time. MuseTalk's live mode prepares the face once and then runs much faster;
  a live website needs < 1× on the server GPU.
* The head stays still here (the input is a still photo). A short real video of the avatar (head moving, blinking)
  as input gives the most natural result — that video can be made once with an image-to-video model.